|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Tensor parallelism<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: two shardings, one collective<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
import torch

torch.manual_seed(0)
D, H, RANKS = 256, 1024, 4
x  = torch.randn(8, D)
W1 = torch.randn(D, H)/D**0.5
W2 = torch.randn(H, D)/H**0.5
reference = torch.relu(x @ W1) @ W2
print('reference', tuple(reference.shape))

reference (8, 256)


Shard an MLP across four ranks two different ways, both correct, and count
what each one costs in conversation.

All on the CPU. Stage 20 does it with real collectives; this is about the
design rule, which is the part you have to get right before the collectives
exist.

# Exercise 1: column, then row

In [2]:
def mlp_parallel(x, W1, W2, ranks):
  W1s = list(W1.chunk(ranks, dim=1))       # COLUMNS
  W2s = list(W2.chunk(ranks, dim=0))       # ROWS, matching
  partials = [torch.relu(x @ W1s[r]) @ W2s[r] for r in range(ranks)]
  return sum(partials), 1                  # one all-reduce

out, collectives = mlp_parallel(x, W1, W2, RANKS)
print(f'max difference {(out-reference).abs().max().item():.2e}')
print(f'collectives per block: {collectives}')

max difference 2.38e-07
collectives per block: 1


# Exercise 2: the other way round

Also correct. Count the collectives.

In [3]:
def mlp_wrong(x, W1, W2, ranks):
  """Row-parallel first. Now the hidden activations must be gathered
  BEFORE relu, because each rank holds a partial sum of them."""
  W1s = list(W1.chunk(ranks, dim=0))       # rows: shards the INPUT
  xs  = list(x.chunk(ranks, dim=1))
  h = sum(xs[r] @ W1s[r] for r in range(ranks))   # collective 1
  h = torch.relu(h)
  W2s = list(W2.chunk(ranks, dim=0))
  hs  = list(h.chunk(ranks, dim=1))
  out = sum(hs[r] @ W2s[r] for r in range(ranks)) # collective 2
  return out, 2

out2, c2 = mlp_wrong(x, W1, W2, RANKS)
print(f'max difference {(out2-reference).abs().max().item():.2e}   (still correct)')
print(f'collectives per block: {c2}   <- twice the talking, same answer')

max difference 9.54e-07   (still correct)
collectives per block: 2   <- twice the talking, same answer


# Exercise 3: what a collective costs, without a clock

Both sides of this comparison are bytes divided by a bandwidth, so the
milliseconds cancel. Write the ratio of collective time to compute time and
you will find the layer count cancels too.

What is left is your model, your batch, and exactly one number about the
machine: how many times faster its memory is than its interconnect.

In [4]:
def comm_over_compute(batch, d_model, ranks, bw_ratio, collectives=1):
  """Collective time divided by compute time. Dimensionless: no clock in it.

       comm      B C (R-1)     HBM
       ----  =  -----------  x -----
       step        6 d           I
  """
  return batch * collectives * (ranks-1) / (6*d_model) * bw_ratio

BW_RATIO = 25.0        # PCIe node: HBM is about 25x the interconnect

print(f"{'batch':>6} {'1 collective':>14} {'2 collectives':>15}")
for b in (1, 32, 256):
  one = comm_over_compute(b, 4096, 8, BW_RATIO, 1)
  two = comm_over_compute(b, 4096, 8, BW_RATIO, 2)
  print(f'{b:>6} {one:>14.3f} {two:>15.3f}')
print('\n(ratio of talking to computing. 1.0 means half your step is the network.)')

for ratio, name in ((1.0,'NVLink'), (25.0,'PCIe'), (200.0,'Ethernet')):
  b1 = 6*4096/((8-1)*ratio*1)
  b2 = 6*4096/((8-1)*ratio*2)
  print(f'{name:>9}: talking overtakes computing at batch {b1:>7,.0f} with 1 '
        f'collective, {b2:>7,.0f} with 2')

 batch   1 collective   2 collectives
     1          0.007           0.014
    32          0.228           0.456
   256          1.823           3.646

(ratio of talking to computing. 1.0 means half your step is the network.)
   NVLink: talking overtakes computing at batch   3,511 with 1 collective,   1,755 with 2
     PCIe: talking overtakes computing at batch     140 with 1 collective,      70 with 2
 Ethernet: talking overtakes computing at batch      18 with 1 collective,       9 with 2


### Both arrangements are correct. Only one is usable.

Column-then-row needs one all-reduce per block. Row-then-column needs
two, because `relu` is not linear and a partial sum cannot be passed
through it.

That is the whole design rule, and it generalises: **put the collective
where the non-linearity is not**. Attention splits the same way for the
same reason, whole heads per rank, because softmax reduces over a row
and a rank must own the whole row.

### And the number that decides whether any of this is worth it

At batch 1 the collectives are a rounding error even on PCIe. At batch
256 with eight ranks they are most of the step, and doubling them takes
you past it.

So tensor parallelism and continuous batching pull against each other.
Every plot in Part 2 said to raise the batch; this one says the
interconnect gets a vote. On NVLink it does not care. On PCIe it decides
your maximum batch size, which decides your throughput, which was the
thing you split the model to get.

Stage 20 builds it with real collectives and checks that the two-rank
output matches the one-rank output exactly. The lesson is where the
communication lands, not the speed: on one GPU there is none to be had.

    ./vc guide 20